# Noise, aliasing, and observable information

Compare independently trained temporal encoders under low noise, higher noise, and a deliberately ambiguous second sensor. For every sensor setting, contrast local dynamics with its isospectral scrambled control.

The ambiguous sensor maps antipodal states to exactly the same observation distribution. A single-frame encoder cannot distinguish these states, regardless of training. We evaluate both the original circular coordinates and their observable doubled-angle quotient; the latter avoids demanding impossible recovery.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_NAME = 'The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning'
REPO_REF = 'main'  # Pin an immutable commit for reproducible collection.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / REPO_NAME]
ROOT = next((p for p in candidates if (p / 'src/dynalign/worlds.py').exists()), None)
if ROOT is None:
    ROOT = Path.cwd() / REPO_NAME
    subprocess.check_call(['git', 'clone', REPO_URL, str(ROOT)])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', REPO_REF])
sys.path.insert(0, str(ROOT / 'src'))
try:
    import numpy, scipy, pandas, matplotlib, torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.experiments import environment, grid, run_grid
from dynalign.reporting import contrast_summary, plot_results
from dynalign.worlds import sensor_table, circle_coordinates
print(environment())

## Configuration and matched comparisons
Changing noise or aliasing changes available information. Therefore the exact information-matching argument applies **within each sensor setting** to local versus scrambled dynamics, not between clean and ambiguous settings. Do not interpret the noise sweep as an information-controlled effect of dynamics.

The encoders have no history input. This is a boundary test, not a recurrent state-estimation experiment. The same objective and update budget are used throughout. `smoke` is only an execution check. Use durable storage for checkpoints if the cloud VM may terminate.

In [ ]:
PROFILE = os.environ.get('DYNALIGN_PROFILE', 'smoke')
DEVICE = os.environ.get('DYNALIGN_DEVICE', 'auto')
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT
OUTPUT_DIR = Path(os.environ.get('DYNALIGN_OUTPUT_DIR', str(base / 'outputs'))) / '03_noise_and_observability'
specs = grid('robustness', PROFILE)
display(pd.DataFrame(specs))
aliased = sensor_table(32, 123, aliased=True)
np.testing.assert_array_equal(aliased[:16], aliased[16:])
np.testing.assert_allclose(circle_coordinates(32, 2)[:16], circle_coordinates(32, 2)[16:], atol=1e-12)
print('Exact antipodal aliasing and observable quotient verified.')

## Train separate encoders
No common latent target or paired cross-modal training examples are supplied. Evaluation and calibration use independent noise draws on a balanced set of known states. Fixed linear probes are fitted after freezing the encoders. A high-dimensional flexible probe could memorize state IDs, so it is intentionally not used.

In [ ]:
results = run_grid(specs, OUTPUT_DIR, DEVICE)
display(results[['case', 'condition', 'world', 'repeat', 'cka', 'procrustes_error', 'any_collapse', 'training_seconds']])

## Geometry and accessible factors
Read the alignment and probe measurements together. Low recovery of the original coordinates by the aliased sensor is expected from the observation construction and cannot by itself diagnose poor optimization. The doubled-angle quotient is accessible in that sensor. Neither a probe nor CKA establishes causal identification.

Initialization repeats are nested within worlds. No uncertainty interval is produced with fewer than five valid worlds. Collapsed runs remain in the output and must be reported.

In [ ]:
contrasts = contrast_summary(results, 'cka')
contrasts.to_csv(OUTPUT_DIR / 'paired_contrasts.csv', index=False)
display(contrasts)
display(results[['case', 'condition', 'world', 'coordinate_r2_a', 'coordinate_r2_b', 'quotient_r2_a', 'quotient_r2_b', 'effective_rank_a', 'effective_rank_b', 'any_collapse']])
figure = plot_results(results, 'cka')
figure.savefig(OUTPUT_DIR / 'observability_alignment.png', dpi=150)
plt.show()

## Export
These are retraining comparisons, not test-time corruption or out-of-distribution adaptation claims. Any difference is conditional on the specified sensor family and single-frame encoders. Preserve all settings and diagnostics, including missing values and unsuccessful runs.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('Saved archive:', archive)
display(FileLink(archive))